# Gold layer

The business view: one wide row per hour, joining weather, consumption, wind
production and price into a single table an analyst can query without knowing
anything about the sources.

Silver made the four sources describe the same world in the same way. Gold puts
them side by side and decides what the question actually is.

## Design decisions

**Weather is pivoted, not averaged.** Silver keeps four rows per hour, one per
observation point. A join needs one row per hour, so the four points become
eight columns. Averaging them away would have destroyed the regional signal:
Vaasa sits on the west coast where most of Finland's wind capacity is, and its
weather is the one most likely to explain wind production. A national mean cannot
answer that question.

**National means are carried as well.** Some questions are about the country
rather than a region. Temperature driving heating demand is national in
character; wind driving turbine output is not. Both shapes are present so
neither question requires a second table.

**The join is an inner join.** The four sources do not cover identical windows.
Fingrid was extracted with a rolling 12-month window while weather and price used
a fixed one. An inner join trims the result to the period all four cover, without
any hand-written date bounds. An outer join would have produced rows where half
the columns are empty, which is worse than not having the row.

## Coverage

`2025-09-23 15:00` to `2026-09-17 23:00` UTC, 8,624 rows.

The start is set by Fingrid, the end by weather. This is the window in which every
statement made from this table is supported by all four sources.

## One hour is missing, and that is correct

The window spans 8,625 hourly points but the table has 8,624.

`2026-06-04 12:00` UTC is absent. Fingrid's wind production series is missing the
12:15 reading for that hour, so silver dropped the hour as incomplete rather than
averaging three quarters and presenting the result as equal to every other hour.
The inner join then removed that hour from all four sources.

The gap was found by tracing a single row count discrepancy through the pipeline:
`bronze_wind` held one reading fewer than `bronze_consumption`, `silver_wind` one
hour fewer, and gold one row fewer than the window implies. A `LAG` window
function over `silver_wind` located the exact hour.

This is the pipeline behaving as intended. The alternative would have been a
silent error: one hour whose average is computed differently from the other
8,623, with nothing to indicate it.

## Columns

| Column | Meaning |
| --- | --- |
| `time_utc` | Hour start, UTC. The join key and the source of truth. |
| `time_local` | Same instant in Europe/Helsinki, for reporting. |
| `FI_S_temp_c`, `FI_S_wind_ms` | Helsinki, south |
| `FI_W_temp_c`, `FI_W_wind_ms` | Vaasa, west coast, near most wind capacity |
| `FI_E_temp_c`, `FI_E_wind_ms` | Kuopio, east |
| `FI_N_temp_c`, `FI_N_wind_ms` | Oulu, north |
| `temp_avg_c`, `wind_avg_ms` | Unweighted mean across the four points |
| `consumption_mw` | Hourly mean electricity consumption, Finland |
| `wind_mw` | Hourly mean wind power generation, Finland |
| `price_eur_mwh` | Hourly mean day-ahead price, Finnish bidding zone |

## What this table does not establish

Any relationship visible here is **correlation, not causation**. Weather,
consumption and price move together for reasons this dataset cannot separate:
time of day, day of week, season, industrial activity, interconnector flows and
the behaviour of other Nordic bidding zones all act at once and none of them are
present as columns.

The four weather points are a deliberate simplification. Finland has 19 regions;
this uses four cities chosen to span the north/south temperature range and to
put one observation near the west coast wind capacity. It is not an
administrative or population-weighted division and should not be described as one.

In [0]:
from pyspark.sql import functions as F

SCHEMA = "workspace.energy_weather"

# Narrow each source to its join key and its measures
consumption = spark.table(f"{SCHEMA}.silver_consumption").select(
    "time_utc", "consumption_mw"
)
wind = spark.table(f"{SCHEMA}.silver_wind").select("time_utc", "wind_mw")
price = spark.table(f"{SCHEMA}.silver_price").select("time_utc", "price_eur_mwh")

weather = spark.table(f"{SCHEMA}.silver_weather")

# One row per hour, one column pair per observation point.
# The explicit value list keeps column order stable and lets Spark
# skip a scan it would otherwise need to discover the values.
weather_wide = (
    weather.groupBy("time_utc", "time_local")
    .pivot("area_id", ["FI_S", "FI_W", "FI_E", "FI_N"])
    .agg(
        F.first("temperature_c").alias("temp_c"),
        F.first("wind_speed_ms").alias("wind_ms"),
    )
)

# National means, for questions that are about the country rather than a region
weather_avg = weather.groupBy("time_utc").agg(
    F.avg("temperature_c").alias("temp_avg_c"),
    F.avg("wind_speed_ms").alias("wind_avg_ms"),
)

# Inner joins trim the result to the window all four sources cover
gold_hourly = (
    weather_wide.join(weather_avg, on="time_utc", how="inner")
    .join(consumption, on="time_utc", how="inner")
    .join(wind, on="time_utc", how="inner")
    .join(price, on="time_utc", how="inner")
)

print("Rows:", gold_hourly.count())
gold_hourly.select(
    F.min("time_utc").alias("first"), F.max("time_utc").alias("last")
).show(truncate=False)
gold_hourly.printSchema()

In [0]:
gold_hourly.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.energy_weather.gold_hourly"
)

print("Written:", spark.table("workspace.energy_weather.gold_hourly").count())

In [0]:
%sql
WITH with_previous AS (
  SELECT
    time_utc,
    LAG(time_utc) OVER (ORDER BY time_utc) AS previous_time
  FROM workspace.energy_weather.silver_wind
)
SELECT
  previous_time,
  time_utc,
  timestampdiff(HOUR, previous_time, time_utc) AS gap_hours
FROM with_previous
WHERE timestampdiff(HOUR, previous_time, time_utc) <> 1

In [0]:
%sql
SELECT startTime, value
FROM workspace.energy_weather.bronze_wind
WHERE startTime >= '2026-06-04 11:00:00'
  AND startTime <  '2026-06-04 14:00:00'
ORDER BY startTime